In [ ]:
import pandas as pd
import os
import joblib
from sklearn import metrics
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

In [ ]:
def calculate_auc_ci(y_true, y_scores, confidence_level=0.95):
    fpr_values, tpr_values, _ = roc_curve(y_true, y_scores)
    auc_value = auc(fpr_values, tpr_values)

    n_bootstraps = 1000
    rng = np.random.RandomState(42)
    bootstrapped_scores = []
    
    for i in range(n_bootstraps):
        indices = rng.randint(0, len(y_scores), len(y_scores))
        if len(np.unique(y_true[indices])) < 2:
            continue
        fpr_bootstrap, tpr_bootstrap, _ = roc_curve(y_true[indices], y_scores[indices])
        score = auc(fpr_bootstrap, tpr_bootstrap)
        bootstrapped_scores.append(score)
        
    sorted_scores = np.array(bootstrapped_scores)
    sorted_scores.sort()
    confidence_lower = sorted_scores[int((1 - confidence_level) / 2 * len(sorted_scores))]
    confidence_upper = sorted_scores[int((1 + confidence_level) / 2 * len(sorted_scores))]
    
    return auc_value, confidence_lower, confidence_upper

In [ ]:
valid_path = 'path\internal_valid_PCA.xlsx' # prospective_valid_PCA.xlsx & external_valid_PCA.xlsx 
valid_df = pd.read_excel(valid_path,sheet_name='Sheet1')
valid_features = valid_df.iloc[:, 1:] 
valid_labels = valid_df.iloc[:,0] 

In [ ]:
save_dir = 'path\'
os.makedirs(save_dir, exist_ok=True)

In [ ]:
best_model_path = 'path\best_model.pkl'
best_model = joblib.load(best_model_path)

valid_y_proba = best_model.predict_proba(valid_features)[:, 1]  
valid_fpr, valid_tpr, thresholds = roc_curve(valid_labels, valid_y_proba)

# Model trainging : best_threshold
valid_y_pred = (valid_y_proba >= best_threshold).astype(int)

results_df = pd.DataFrame({'Actual': valid_labels, 'Predicted': valid_y_pred, 'Probability': valid_y_proba})
results_df.to_excel(os.path.join(save_dir,'prediction_Internal_valid.xlsx'), index=False)

valid_accuracy = metrics.accuracy_score(valid_labels, valid_y_pred)
valid_precision, valid_recall, valid_fscore, _ = metrics.precision_recall_fscore_support(valid_labels, valid_y_pred, average='binary')
valid_specificity = metrics.recall_score(valid_labels, valid_y_pred, pos_label=0)
valid_roc_auc = metrics.roc_auc_score(valid_labels, valid_y_proba)

results_dict = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'Specificity', 'ROC AUC'],
    'Best_Threshold': [valid_accuracy, valid_precision, valid_recall, valid_fscore, 
                       valid_specificity, valid_roc_auc]
}
results_df = pd.DataFrame(results_dict)
results_df.to_excel(os.path.join(save_dir,'Internal_valid_metrics_results.xlsx'), index=False)

In [ ]:
auc_value, auc_ci_lower, auc_ci_upper = calculate_auc_ci(valid_labels, valid_y_proba)

plt.figure(facecolor='white', figsize=(4, 4))
plt.plot(valid_fpr, valid_tpr, color='#8ECFC9', alpha=1, 
         label=f"AUC = {auc_value:.3f} \n(95% CI [{auc_ci_lower:.3f}, {auc_ci_upper:.3f}])", linewidth=1.5)
plt.fill_between(valid_fpr, valid_tpr, color='#8ECFC9', alpha=0.1) 

plt.xlabel('False Positive Rate (1-Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
plt.plot([0, 1], [0, 1], color='black', linestyle='--')
plt.legend(fontsize=10, loc='lower right', frameon=False)
plt.gca().set_aspect('equal', adjustable='box')
plt.xlim([0, 1])
plt.ylim([0, 1])
ax = plt.gca()
ax.spines['top'].set_linewidth(1)
ax.spines['bottom'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)
ax.spines['right'].set_linewidth(1)
ax.tick_params(axis='both', width=1) 
plt.xticks(fontsize=12)  
plt.yticks(fontsize=12)  
plt.savefig(os.path.join(save_dir, 'ROC_Internal_Valid.pdf'), dpi=300)
plt.show()